# GitLab Issues Fetcher - One Row Per Issue

## Key Feature: Consolidated Data Structure
- **One row per issue_id**
- Multiple links consolidated with **comma-separated values**
- No duplicate issue rows

This notebook:
1. Fetches all issues from IKG and SWAT GitLab projects
2. Consolidates multiple links per issue into comma-separated strings
3. Creates Excel file with separate worksheets for IKG and SWAT
4. Loads data into Greenplum database tables
5. Maintains archive tables for historical tracking

## 1. Import Libraries

In [ ]:
import requests
import pandas as pd
import getpass
from typing import List, Dict, Any, Tuple
from datetime import datetime
import psycopg2
from psycopg2 import sql
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils.dataframe import dataframe_to_rows
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries imported successfully")

## 2. Configuration

In [ ]:
# GitLab Configuration
GITLAB_URL = 'https://devcloud.ubs.net'
IKG_PROJECT_PATH = 'ubs/gwma/smart-technology-and-analytics/staat-data-science/staat-ds-insights-cl/commons/staat-ds-insights-home'
SWAT_PROJECT_PATH = 'ubs/gwma/smart-technology-and-analytics/staat-data-science/staat-ds-insights-cl/commons/staat-ds-insights-cl-home'

# Greenplum Configuration
GREENPLUM_HOST = 'greenplum-rdsp.zur.swissbank.com'
GREENPLUM_PORT = 5432
GREENPLUM_DB = 'gprdsp'
GREENPLUM_USER = 'ds_rdsp_dev'
GREENPLUM_SCHEMA = 'sandbox_prj_smart_insights'

# Table names
OUTPUT_TABLE1 = 'ikg_issue_details'
OUTPUT_TABLE2 = 'swat_issue_details'
ARCHIVE_TABLE1 = 'ikg_issue_details_archive'
ARCHIVE_TABLE2 = 'swat_issue_details_archive'

print(f"GitLab URL: {GITLAB_URL}")
print(f"IKG Project: {IKG_PROJECT_PATH.split('/')[-1]}")
print(f"SWAT Project: {SWAT_PROJECT_PATH.split('/')[-1]}")
print(f"\nGreenplum: {GREENPLUM_HOST}:{GREENPLUM_PORT}/{GREENPLUM_DB}")
print(f"Schema: {GREENPLUM_SCHEMA}")

## 3. GitLab Authentication

In [ ]:
# Get GitLab Personal Access Token
gitlab_token = getpass.getpass("Enter your GitLab Personal Access Token: ")

headers = {
    'PRIVATE-TOKEN': gitlab_token,
    'Content-Type': 'application/json'
}

print("✓ GitLab authentication configured")

## 4. Helper Functions

In [ ]:
def get_project_id(gitlab_url: str, project_path: str, headers: dict) -> Tuple[str, str]:
    """Get project ID and name from project path"""
    encoded_path = requests.utils.quote(project_path, safe='')
    url = f"{gitlab_url}/api/v4/projects/{encoded_path}"
    
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    project_data = response.json()
    
    return project_data['id'], project_data['name']


def fetch_issue_links(gitlab_url: str, project_id: str, issue_iid: int, headers: dict) -> List[Dict]:
    """Fetch all links for a specific issue"""
    url = f"{gitlab_url}/api/v4/projects/{project_id}/issues/{issue_iid}/links"
    
    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        return response.json()
    except:
        return []


def fetch_all_issues(gitlab_url: str, project_id: str, project_name: str, headers: dict) -> List[Dict]:
    """Fetch all issues with linkage details"""
    all_issues = []
    page = 1
    per_page = 100
    
    url = f"{gitlab_url}/api/v4/projects/{project_id}/issues"
    
    print(f"Fetching issues from {project_name}...")
    
    while True:
        params = {
            'per_page': per_page,
            'page': page,
            'state': 'all',
            'scope': 'all',
            'with_labels_details': True
        }
        
        response = requests.get(url, headers=headers, params=params)
        response.raise_for_status()
        issues = response.json()
        
        if not issues:
            break
        
        # Fetch links for each issue
        for issue in issues:
            issue['_links_data'] = fetch_issue_links(gitlab_url, project_id, issue['iid'], headers)
        
        all_issues.extend(issues)
        print(f"  Page {page}: {len(issues)} issues (Total: {len(all_issues)})")
        
        if len(issues) < per_page:
            break
        
        page += 1
    
    print(f"✓ Total issues fetched: {len(all_issues)}")
    return all_issues

print("✓ Helper functions defined")

In [ ]:
def extract_issue_data(issues: List[Dict], project_identifier: str) -> pd.DataFrame:
    """
    Extract comprehensive issue data with CONSOLIDATED linkage details.
    ONE ROW PER ISSUE - links are comma-separated.
    """
    extracted_data = []
    current_datetime = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    
    for issue in issues:
        assignees = [a.get('name', '') for a in issue.get('assignees', [])]
        assignee_str = ', '.join(assignees) if assignees else None
        
        labels = issue.get('labels', [])
        label_str = ', '.join(labels) if labels else None
        
        iteration = issue.get('iteration', {}).get('title', '') if issue.get('iteration') else None
        epic = issue.get('epic', {}).get('title', '') if issue.get('epic') else None
        epic_iid = issue.get('epic', {}).get('iid', '') if issue.get('epic') else None
        
        participants = [p.get('name', '') for p in issue.get('participants', [])]
        participants_str = ', '.join(participants) if participants else None
        
        milestone = issue.get('milestone', {}).get('title', '') if issue.get('milestone') else None
        
        time_estimate = issue.get('time_stats', {}).get('time_estimate')
        time_spent = issue.get('time_stats', {}).get('total_time_spent')
        
        task_completion = None
        if issue.get('task_completion_status'):
            completed = issue['task_completion_status'].get('completed_count', 0)
            total = issue['task_completion_status'].get('count', 0)
            task_completion = f"{completed}/{total}"
        
        # CONSOLIDATE LINKS - collect all into comma-separated strings
        links_data = issue.get('_links_data', [])
        
        link_ids = []
        link_issue_ids = []
        link_issue_iids = []
        link_types = []
        link_urls = []
        link_issue_titles = []
        linked_project_ids = []
        
        for link in links_data:
            if link.get('id'):
                link_ids.append(str(link.get('id', '')))
            if link.get('issue_link_id'):
                link_issue_ids.append(str(link.get('issue_link_id', '')))
            if link.get('iid'):
                link_issue_iids.append(str(link.get('iid', '')))
            if link.get('link_type'):
                link_types.append(str(link.get('link_type', '')))
            if link.get('web_url'):
                link_urls.append(str(link.get('web_url', '')))
            if link.get('title'):
                link_issue_titles.append(str(link.get('title', '')))
            if link.get('project_id'):
                linked_project_ids.append(str(link.get('project_id', '')))
        
        # ONE ROW PER ISSUE with comma-separated link values
        issue_data = {
            'project': project_identifier,
            'issue_id': issue.get('id'),
            'issue_iid': issue.get('iid'),
            'title': issue.get('title'),
            'description': issue.get('description'),
            'state': issue.get('state'),
            'web_url': issue.get('web_url', ''),
            'link_id': ', '.join(link_ids) if link_ids else None,
            'link_issue_id': ', '.join(link_issue_ids) if link_issue_ids else None,
            'link_issue_iid': ', '.join(link_issue_iids) if link_issue_iids else None,
            'link_type': ', '.join(link_types) if link_types else None,
            'link_url': ', '.join(link_urls) if link_urls else None,
            'link_issue_title': ', '.join(link_issue_titles) if link_issue_titles else None,
            'linked_project_id': ', '.join(linked_project_ids) if linked_project_ids else None,
            'author': issue.get('author', {}).get('name'),
            'author_username': issue.get('author', {}).get('username'),
            'created_by_id': issue.get('author', {}).get('id'),
            'assignee': assignee_str,
            'assignee_ids': ', '.join([str(a.get('id', '')) for a in issue.get('assignees', [])]),
            'issue_created_date': issue.get('created_at'),
            'created_at': issue.get('created_at'),
            'updated_at': issue.get('updated_at'),
            'closed_at': issue.get('closed_at'),
            'due_date': issue.get('due_date'),
            'start_date': issue.get('start_date'),
            'current_date_time': current_datetime,
            'labels': label_str,
            'milestone': milestone,
            'iteration': iteration,
            'epic': epic,
            'epic_iid': epic_iid,
            'weight': issue.get('weight'),
            'parent_iid': None,
            'has_tasks': issue.get('has_tasks'),
            'task_completion_status': task_completion,
            'participants': participants_str,
            'upvotes': issue.get('upvotes'),
            'downvotes': issue.get('downvotes'),
            'user_notes_count': issue.get('user_notes_count'),
            'merge_requests_count': issue.get('merge_requests_count'),
            'time_estimate_hours': time_estimate / 3600 if time_estimate else None,
            'time_spent_hours': time_spent / 3600 if time_spent else None,
            'confidential': issue.get('confidential'),
            'discussion_locked': issue.get('discussion_locked'),
            'issue_type': issue.get('issue_type'),
            'severity': issue.get('severity'),
            'health_status': issue.get('health_status'),
        }
        
        extracted_data.append(issue_data)
    
    return pd.DataFrame(extracted_data)

print("✓ Data extraction function defined (one row per issue)")

## 5. Fetch IKG Issues

In [ ]:
# Get IKG project info
ikg_project_id, ikg_project_name = get_project_id(GITLAB_URL, IKG_PROJECT_PATH, headers)
print(f"✓ IKG Project: {ikg_project_name} (ID: {ikg_project_id})")

# Fetch IKG issues
ikg_issues = fetch_all_issues(GITLAB_URL, ikg_project_id, ikg_project_name, headers)

# Extract IKG data
print("\nExtracting IKG issue data...")
ikg_df = extract_issue_data(ikg_issues, 'staat-ds-insights-home')
print(f"✓ IKG data extracted: {len(ikg_df)} rows (one per issue)")
print(f"  Columns: {len(ikg_df.columns)}")

In [ ]:
# Preview IKG data
print("IKG Data Preview:")
print(f"Shape: {ikg_df.shape}")
print(f"\nUnique issues: {ikg_df['issue_id'].nunique()}")
print(f"Total rows: {len(ikg_df)}")
print("\nFirst few rows:")
ikg_df.head()

In [ ]:
# Check for issues with multiple links (comma-separated)
issues_with_links = ikg_df[ikg_df['link_id'].notna()]
print(f"Issues with links: {len(issues_with_links)}")
if len(issues_with_links) > 0:
    print("\nSample issue with links:")
    sample = issues_with_links.iloc[0]
    print(f"Issue #{sample['issue_iid']}: {sample['title']}")
    print(f"Link Types: {sample['link_type']}")
    print(f"Linked Issue IIDs: {sample['link_issue_iid']}")

## 6. Fetch SWAT Issues

In [ ]:
# Get SWAT project info
swat_project_id, swat_project_name = get_project_id(GITLAB_URL, SWAT_PROJECT_PATH, headers)
print(f"✓ SWAT Project: {swat_project_name} (ID: {swat_project_id})")

# Fetch SWAT issues
swat_issues = fetch_all_issues(GITLAB_URL, swat_project_id, swat_project_name, headers)

# Extract SWAT data
print("\nExtracting SWAT issue data...")
swat_df = extract_issue_data(swat_issues, 'staat-ds-insights-cl-home')
print(f"✓ SWAT data extracted: {len(swat_df)} rows (one per issue)")
print(f"  Columns: {len(swat_df.columns)}")

In [ ]:
# Preview SWAT data
print("SWAT Data Preview:")
print(f"Shape: {swat_df.shape}")
print(f"\nUnique issues: {swat_df['issue_id'].nunique()}")
print(f"Total rows: {len(swat_df)}")
print("\nFirst few rows:")
swat_df.head()

## 7. Create Excel File with Multiple Sheets

In [ ]:
def create_excel_with_sheets(ikg_df: pd.DataFrame, swat_df: pd.DataFrame, filename: str):
    """Create Excel file with IKG and SWAT worksheets"""
    wb = Workbook()
    wb.remove(wb.active)
    
    # Create IKG sheet
    ikg_sheet = wb.create_sheet('IKG')
    for r in dataframe_to_rows(ikg_df, index=False, header=True):
        ikg_sheet.append(r)
    
    # Format IKG header
    for cell in ikg_sheet[1]:
        cell.font = Font(bold=True, color='FFFFFF')
        cell.fill = PatternFill(start_color='366092', end_color='366092', fill_type='solid')
        cell.alignment = Alignment(horizontal='center', vertical='center')
    
    # Auto-adjust column widths
    for column in ikg_sheet.columns:
        max_length = 0
        column_letter = column[0].column_letter
        for cell in column:
            try:
                if len(str(cell.value)) > max_length:
                    max_length = len(str(cell.value))
            except:
                pass
        adjusted_width = min(max_length + 2, 50)
        ikg_sheet.column_dimensions[column_letter].width = adjusted_width
    
    # Create SWAT sheet
    swat_sheet = wb.create_sheet('SWAT')
    for r in dataframe_to_rows(swat_df, index=False, header=True):
        swat_sheet.append(r)
    
    # Format SWAT header
    for cell in swat_sheet[1]:
        cell.font = Font(bold=True, color='FFFFFF')
        cell.fill = PatternFill(start_color='366092', end_color='366092', fill_type='solid')
        cell.alignment = Alignment(horizontal='center', vertical='center')
    
    # Auto-adjust column widths
    for column in swat_sheet.columns:
        max_length = 0
        column_letter = column[0].column_letter
        for cell in column:
            try:
                if len(str(cell.value)) > max_length:
                    max_length = len(str(cell.value))
            except:
                pass
        adjusted_width = min(max_length + 2, 50)
        swat_sheet.column_dimensions[column_letter].width = adjusted_width
    
    wb.save(filename)
    print(f"✓ Excel file saved: {filename}")

# Create Excel file
excel_filename = 'gitlab_issues_ikg_swat.xlsx'
create_excel_with_sheets(ikg_df, swat_df, excel_filename)

## 8. Greenplum Database Connection

In [ ]:
# Get database password
db_password = getpass.getpass("Enter Greenplum database password: ")

# Connect to Greenplum
try:
    conn = psycopg2.connect(
        host=GREENPLUM_HOST,
        port=GREENPLUM_PORT,
        database=GREENPLUM_DB,
        user=GREENPLUM_USER,
        password=db_password
    )
    conn.autocommit = False
    print(f"✓ Connected to Greenplum: {GREENPLUM_DB}")
except Exception as e:
    print(f"✗ Connection error: {e}")
    raise

## 9. Database Helper Functions

In [ ]:
def create_table(conn, schema: str, table_name: str, df: pd.DataFrame, drop_if_exists: bool = True):
    """Create table in Greenplum"""
    cursor = conn.cursor()
    
    try:
        if drop_if_exists:
            drop_query = f"DROP TABLE IF EXISTS {schema}.{table_name}"
            cursor.execute(drop_query)
            print(f"  Dropped {schema}.{table_name} (if existed)")
        
        columns = []
        for col, dtype in df.dtypes.items():
            if dtype == 'object':
                col_type = 'TEXT'
            elif dtype == 'int64':
                col_type = 'BIGINT'
            elif dtype == 'float64':
                col_type = 'DOUBLE PRECISION'
            elif dtype == 'bool':
                col_type = 'BOOLEAN'
            else:
                col_type = 'TEXT'
            columns.append(f"{col} {col_type}")
        
        create_query = f"""
            CREATE TABLE {schema}.{table_name} (
                {', '.join(columns)}
            ) DISTRIBUTED RANDOMLY
        """
        
        cursor.execute(create_query)
        conn.commit()
        print(f"  ✓ Created {schema}.{table_name}")
    except Exception as e:
        conn.rollback()
        print(f"  ✗ Error: {e}")
        raise
    finally:
        cursor.close()


def insert_data(conn, schema: str, table_name: str, df: pd.DataFrame):
    """Insert data into Greenplum table"""
    cursor = conn.cursor()
    
    try:
        columns = df.columns.tolist()
        placeholders = ', '.join(['%s'] * len(columns))
        columns_str = ', '.join(columns)
        
        insert_query = f"""
            INSERT INTO {schema}.{table_name} ({columns_str})
            VALUES ({placeholders})
        """
        
        data = [tuple(x) for x in df.to_numpy()]
        cursor.executemany(insert_query, data)
        conn.commit()
        
        print(f"  ✓ Inserted {len(df)} rows into {schema}.{table_name}")
    except Exception as e:
        conn.rollback()
        print(f"  ✗ Error: {e}")
        raise
    finally:
        cursor.close()

print("✓ Database functions defined")

## 10. Create Main Tables (Drop and Create)

In [ ]:
print("Creating main tables (drop and create)...")

# IKG main table
create_table(conn, GREENPLUM_SCHEMA, OUTPUT_TABLE1, ikg_df, drop_if_exists=True)

# SWAT main table
create_table(conn, GREENPLUM_SCHEMA, OUTPUT_TABLE2, swat_df, drop_if_exists=True)

print("\n✓ Main tables created")

## 11. Create Archive Tables (Create Only If Not Exist)

In [ ]:
print("Creating archive tables (if not exist)...")

# Check and create IKG archive
cursor = conn.cursor()
cursor.execute(f"""
    SELECT EXISTS (
        SELECT 1 FROM information_schema.tables 
        WHERE table_schema = '{GREENPLUM_SCHEMA}' 
        AND table_name = '{ARCHIVE_TABLE1}'
    )
""")
ikg_archive_exists = cursor.fetchone()[0]
cursor.close()

if not ikg_archive_exists:
    create_table(conn, GREENPLUM_SCHEMA, ARCHIVE_TABLE1, ikg_df, drop_if_exists=False)
else:
    print(f"  Archive table {GREENPLUM_SCHEMA}.{ARCHIVE_TABLE1} already exists")

# Check and create SWAT archive
cursor = conn.cursor()
cursor.execute(f"""
    SELECT EXISTS (
        SELECT 1 FROM information_schema.tables 
        WHERE table_schema = '{GREENPLUM_SCHEMA}' 
        AND table_name = '{ARCHIVE_TABLE2}'
    )
""")
swat_archive_exists = cursor.fetchone()[0]
cursor.close()

if not swat_archive_exists:
    create_table(conn, GREENPLUM_SCHEMA, ARCHIVE_TABLE2, swat_df, drop_if_exists=False)
else:
    print(f"  Archive table {GREENPLUM_SCHEMA}.{ARCHIVE_TABLE2} already exists")

print("\n✓ Archive tables ready")

## 12. Insert Data into Main Tables

In [ ]:
print("Inserting data into main tables...")

insert_data(conn, GREENPLUM_SCHEMA, OUTPUT_TABLE1, ikg_df)
insert_data(conn, GREENPLUM_SCHEMA, OUTPUT_TABLE2, swat_df)

print("\n✓ Data inserted into main tables")

## 13. Insert Data into Archive Tables

In [ ]:
print("Inserting data into archive tables...")

insert_data(conn, GREENPLUM_SCHEMA, ARCHIVE_TABLE1, ikg_df)
insert_data(conn, GREENPLUM_SCHEMA, ARCHIVE_TABLE2, swat_df)

print("\n✓ Data inserted into archive tables")

## 14. Verify Data Load

In [ ]:
# Verify row counts
cursor = conn.cursor()

print("Verifying data load...\n")

for table in [OUTPUT_TABLE1, OUTPUT_TABLE2, ARCHIVE_TABLE1, ARCHIVE_TABLE2]:
    cursor.execute(f"SELECT COUNT(*) FROM {GREENPLUM_SCHEMA}.{table}")
    count = cursor.fetchone()[0]
    print(f"  {GREENPLUM_SCHEMA}.{table}: {count} rows")

cursor.close()
print("\n✓ Data verification complete")

## 15. Close Database Connection

In [ ]:
conn.close()
print("✓ Database connection closed")

## 16. Final Summary

In [ ]:
print("=" * 80)
print("FINAL SUMMARY")
print("=" * 80)
print(f"\nData Structure: ONE ROW PER ISSUE")
print(f"Link Handling: Comma-separated values")
print(f"\nIKG Issues: {len(ikg_df)} rows (unique issues)")
print(f"SWAT Issues: {len(swat_df)} rows (unique issues)")
print(f"\nExcel File: {excel_filename}")
print(f"  - IKG sheet: {len(ikg_df)} rows")
print(f"  - SWAT sheet: {len(swat_df)} rows")
print(f"\nGreenplum Tables:")
print(f"  Main Tables:")
print(f"    - {GREENPLUM_SCHEMA}.{OUTPUT_TABLE1}")
print(f"    - {GREENPLUM_SCHEMA}.{OUTPUT_TABLE2}")
print(f"  Archive Tables:")
print(f"    - {GREENPLUM_SCHEMA}.{ARCHIVE_TABLE1}")
print(f"    - {GREENPLUM_SCHEMA}.{ARCHIVE_TABLE2}")
print(f"\n✓ Process completed successfully!")
print("\nNote: Multiple link values are separated by commas (,)")
print("=" * 80)